# prefit guide: decision-driven EDA on Red Wine Quality

This notebook is a hands-on guide to [prefit](https://github.com/hkln4/prefit). It runs the library end to end on a real dataset, one step at a time, and every step is decided from the output of the step before it.

**What prefit does.** Before you train, it profiles your development data and turns what it measures into a short list of **decisions**: how to split, which columns to question, what to preprocess, which metric to trust. Each decision shows the evidence behind it, the threshold it was compared with, and how uncertain the measurement is. Decisions that depend on your model are tested with small, bounded experiments instead of being assumed.

**What it does not do.** It is not AutoML. It never picks a model, never tunes hyperparameters, and never applies an assumption you have not confirmed.

**Why this dataset.** Red Wine Quality looks like one of the cleanest datasets on Kaggle: 1,599 rows, 11 numeric columns, no missing values. It still hides a problem that a standard train/test split does not catch, and that changes the reported score by a wide margin.

| step | what we do | prefit API |
|---|---|---|
| 1 | Install and load the data | |
| 2 | Set the test data aside, before looking at anything | `pf.split` |
| 3 | First analysis: read the decisions | `pf.analyze`, `decisions_table()` |
| 4 | Answer the questions only we can answer | `report.change_task`, `report.answer` |
| 5 | Read the duplicate finding and record our decision | `report.decision`, `report.resolve` |
| 6 | Standard practice | `Decision.standard` |
| 7 | Test the conditional decision with our model | `report.verify` |
| 8 | Export the model recipe and train | `report.export`, `report.cv_estimate` |
| 9 | Score once on the untouched test set | `pf.review` |
| 10 | Compare with a plain scikit-learn workflow | |
| 11 | Save the HTML report | `report.to_html` |
| 12 | Optional: does an LLM help? | `report.llm_payload`, `report.interpret` |

*Numbers quoted in the text come from the run saved with this notebook (prefit 0.3.1, scikit-learn 1.9.1). Another scikit-learn version can move the random forest's scores in the last digit; the code cells always print the values of your own run.*

## 1. Setup
The core of prefit needs only `pandas`, `numpy` and `pydantic`. This guide uses two extras:

- `verify`: scikit-learn and scipy, for experiments and the exported model recipe
- `report`: plotly, so the HTML report also works offline

> **On Kaggle:** turn on **Settings → Internet** before running the install cell.

In [ ]:
%pip install -q --disable-pip-version-check "prefit[verify,report]>=0.3.1"

In [2]:
import glob
import warnings

import numpy as np
import pandas as pd
import sklearn

import prefit as pf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

SEED = 42
TARGET = "quality"

print(f"prefit       {pf.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"pandas       {pd.__version__}")

prefit       0.3.1
scikit-learn 1.9.1
pandas       3.1.0


### The data
Red variants of the Portuguese *Vinho Verde* wine (Cortez et al., 2009). Each row is one wine: 11 physicochemical lab measurements and a `quality` score from 0 to 10 given by expert tasters.

The loader uses the Kaggle input when it is attached (*Add Input → "Red Wine Quality"* by UCI Machine Learning), then a local copy, and otherwise downloads the original file from the UCI repository. The Kaggle copy is comma-separated and the UCI copy uses semicolons, so pandas sniffs the separator.

In [3]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

candidates = glob.glob("/kaggle/input/**/winequality-red.csv", recursive=True) + glob.glob("winequality-red.csv")
source = candidates[0] if candidates else UCI_URL

df = pd.read_csv(source, sep=None, engine="python")  # the Kaggle copy uses ",", the UCI copy ";"

print(f"source: {source}")
print(f"shape:  {df.shape}")
df.head()

source: winequality-red.csv
shape:  (1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Look at the data as little as possible before splitting: the schema and the target, nothing more. Everything else is prefit's job, and we want to see what it finds on its own.

Notice already that rows 0 and 4 above look identical. We will come back to that.

In [4]:
print(df.dtypes.value_counts().to_string(), end="\n\n")
print(f"missing values: {int(df.isna().sum().sum())}", end="\n\n")
print(df[TARGET].value_counts().sort_index().rename("rows per quality score").to_string())

float64    11
int64       1

missing values: 0

quality
3     10
4     53
5    681
6    638
7    199
8     18


All columns are numeric and nothing is missing. Two things to keep in mind:

- Only six scores occur (3 to 8), and the extremes are rare: 10 wines scored 3 and 18 scored 8.
- The target is an integer with few distinct values. Is this **classification** or **regression**? The data cannot settle that; prefit will ask in step 4.

## 2. Set the test data aside first
Any statistic computed on the test rows leaks into the decisions we make from it, and the final score becomes optimistic. So before computing anything we split off a test set, and **we do not touch it again until step 9**.

`pf.split` has four strategies: `"stratified"` (keep class ratios), `"random"`, `"group"` (whole groups on one side, needs `groups=`) and `"time"` (the latest share by `time_col=`). We stratify on `quality` so the rare scores keep their share on both sides.

> **Shortcut:** `pf.analyze(df, target, holdout=0.2)` sets a holdout aside by itself before profiling, and `report.holdout(df)` gives it back. Splitting explicitly, as here, keeps the test frame visibly separate.

In [5]:
dev_df, test_df = pf.split(df, target=TARGET, test_size=0.2, strategy="stratified", random_state=SEED)

print(f"development rows: {len(dev_df)}")
print(f"test rows:        {len(test_df)}")

pd.DataFrame({
    "all": df[TARGET].value_counts(normalize=True),
    "dev": dev_df[TARGET].value_counts(normalize=True),
    "test": test_df[TARGET].value_counts(normalize=True),
}).sort_index().round(3)

development rows: 1278
test rows:        321


,all,dev,test
quality,,,
3,0.006,0.006,0.006
4,0.033,0.033,0.034
5,0.426,0.426,0.424
6,0.399,0.399,0.399
7,0.124,0.124,0.125
8,0.011,0.011,0.012


The score shares match on all three sides. From here on, everything is computed on `dev_df` only.

`pf.split` also does one thing a plain `train_test_split` does not: if two rows are **exact copies** (all 11 measurements *and* the score), it keeps them on the same side. Checking whether rows are identical uses no statistics about the target, so it is safe to do on the full data. Let us check that claim ourselves:

In [6]:
# A row hash covers every column, target included: equal hashes = identical rows.
hash_dev = pd.util.hash_pandas_object(dev_df, index=False)
hash_test = pd.util.hash_pandas_object(test_df, index=False)

print(f"rows in the full data that have an exact copy: {int(df.duplicated(keep=False).sum())}")
print(f"test rows with an exact copy in development:   {int(hash_test.isin(set(hash_dev)).sum())}")

rows in the full data that have an exact copy: 460
test rows with an exact copy in development:   0


## 3. First analysis: what does the data say?
460 of the 1,599 rows have an identical twin, and still not a single test row has a copy in the development data. Step 5 shows why this matters so much here.

Now the analysis. We call `pf.analyze` on the development data and pass **no model yet**: to choose one we first need the task type, and that is one of the questions we expect prefit to ask. The model comes in at step 7.

Right after the analysis we register the test set with `register_external_holdout`. This stores only metadata (row count, column schema, digests) and counts test rows that have an exact copy in the development data. The test set itself is not profiled.

In [7]:
report = pf.analyze(dev_df, target=TARGET)
report = report.register_external_holdout(test_df, dev_df=dev_df, name="test set")

report  # in a notebook: a short summary card with the next step

<prefit Report r1: classification (inferred) on 'quality' · 1,278 development rows, 11 features · 2 question(s) open, 2 critical/review decision(s). Next: Answer 2 open question(s) with report.answer({...}); contextual decisions stay out of the model recipe until you do.>

In a notebook the report renders as a short card: what is open and what to do next. Here: two questions (task and metric) and two decisions that are critical or need review.

The decisions themselves are easiest to read with `decisions_table()`, one row per decision. Every decision has a **class**, which says how much the data alone can settle it and when it may enter the final model:

| class | meaning | enters the model recipe when |
|---|---|---|
| **procedural** | how to evaluate; correct regardless of the data | always |
| **conditional** | may help or hurt depending on the data and the model | an experiment with *your* model verifies it |
| **contextual** | cannot be decided from data (task type, metric, ...) | you answer the question |
| **review** | needs a human look | never; it is a to-do |

The `standard` column marks **standard practice**: decisions that appear on every dataset (fit preprocessing inside the folds, evaluate on the test set once). They are correct, but they are not news about *this* data, so the reports list them apart.

In [8]:
report.decisions_table()[["class", "status", "standard", "decision", "key evidence"]]

,class,status,standard,decision,key evidence
0,contextual,awaiting_user,False,Treat the task as classification,Task inference = integer target with 6 distinct values (<= 10): could be class labels or small counts
1,procedural,applied,False,Cross-validate with StratifiedKFold (exact duplicates kept in one fold),Rows profiled (dev) = 1278
2,procedural,applied,False,Keep exact duplicates on one side of every split,Exact duplicate rows (extra copies) = 195
3,review,needs_review,False,Decide whether duplicates are the same record or legitimate repeats,Exact duplicate rows (extra copies) = 195
4,procedural,applied,True,Fit all preprocessing inside each fold,Column kinds = numeric=11
5,conditional,proposed,False,Power-transform 2 skewed + 2 near-threshold feature(s),"Skewness · residual sugar = 4.322 [3.532, 4.798]"
6,conditional,proposed,False,Weight classes inversely to frequency,"Imbalance ratio (largest / smallest class) = 68.12 [41.53, 183.4]"
7,contextual,awaiting_user,False,Choose the metric from the cost of errors,Majority-class share (= accuracy of always predicting it) = 0.4264
8,procedural,applied,True,"Evaluate on the test set once, at the end",External holdout (registered) = 321 rows (test set)


Nine decisions. Before going through them, two small helpers that print one decision with its evidence, and the open questions with the effect of each answer:

In [9]:
def show_decision(report, decision_id):
    d = report.decision(decision_id)
    print(d.title)
    print(f"  class: {d.cls.value}   status: {d.status.value}" + ("   (standard practice)" if d.standard else ""))
    print(f"  why:   {d.why_short}")
    print(f"  do:    {d.do_short}")
    if d.risk:
        print(f"  risk:  {d.risk}")
    if d.note:
        print(f"  note:  {d.note}")
    if d.review_note:
        print(f"  your note: {d.review_note}")
    for eid in d.evidence_ids:
        e = report.details.evidence[eid]
        ci = f"  95% CI [{e.ci[0]:.3g}, {e.ci[1]:.3g}]" if e.ci else ""
        print(f"  evidence: {e.label} = {e.value}{ci}")
        print(f"            ({e.method})")


def show_questions(report):
    for q in report.pending_questions():
        print(f"[{q.id}] {q.text}")
        for o in q.options:
            print(f"    {o.value!r:22} {o.label}")
            print(f"    {'':22} -> {o.effect}")

## 4. Answer the questions only we can answer

### 4.1 Task type: classification or regression?

In [10]:
show_decision(report, "d.task")
print()
show_questions(report)

Treat the task as classification
  class: contextual   status: awaiting_user
  why:   Inferred from the target: integer target with 6 distinct values (<= 10): could be class labels or small counts.
  do:    Analyze and evaluate as a classification problem.
  risk:  A wrong task changes every metric and split recommendation. Until you answer, findings and decisions marked as task-dependent assume the inferred task.
  evidence: Task inference = integer target with 6 distinct values (<= 10): could be class labels or small counts
            (dtype + distinct-value heuristic on the target)

[q.task] Is this a classification task? Ordered scores are usually regression; the data cannot tell.
    'classification'       Yes: unordered labels (species, digits, categories)
                           -> keeps the current analysis
    'regression'           No: an ordered score or rating (off by one beats off by three)
                           -> report.change_task(df, 'regression') re-runs the 

prefit inferred *classification* from an integer target with 6 distinct values, and flags it as a guess. The question itself says why it cannot decide: the data cannot tell an ordered score from unordered labels. Digits 0–9 or three grape cultivars coded 0–2 look exactly like this column.

While the task is a guess, some findings exist **only because of the guess**. prefit marks them as *task-dependent*:

In [11]:
print("Only meaningful if the task really is classification:")
for f in report.details.findings:
    if f.task_dependent:
        print(f"  finding  {f.id:22} [{f.severity.value}] {f.message}")
for d in report.decisions:
    if d.task_dependent:
        print(f"  decision {d.id:22} {d.title}")

Only meaningful if the task really is classification:
  finding  f.class_imbalance      [critical] Classes are imbalanced 68.1:1; always predicting the majority class already scores 42.6% accuracy.
  decision d.class_weight         Weight classes inversely to frequency
  decision d.metric               Choose the metric from the cost of errors


The most alarming item in the table, a **critical** class imbalance of 68:1, is one of them. It only means something if `quality` is a set of unordered classes. If we accepted the guess without thinking, we would now be weighting classes and choosing macro F1.

The meaning of the score settles it. `quality` is **ordered**: predicting 6 for a wine rated 5 is a small mistake, predicting 8 is a large one. A classifier treats both mistakes the same; a regressor is penalised more for the larger one. We want the second behaviour, so we treat it as **regression**.

A task change re-runs the analysis (it changes which rules apply), so it is not an `answer()`. `report.change_task(df, ...)` does it on the same, fingerprint-checked frame and keeps what still applies: valid answers, review notes, and the registered test set.

In [12]:
before = {d.id for d in report.decisions}
cv_before = report.decision("d.cv").title

report = report.change_task(dev_df, "regression")

after = {d.id for d in report.decisions}
print(f"task now: {report.task.value}")
print(f"cv before: {cv_before}")
print(f"cv after:  {report.decision('d.cv').title}")
print(f"decisions removed: {sorted(before - after)}")
print(f"decisions added:   {sorted(after - before)}")
print(f"test set still registered: {report.details.manifest.external_holdout['name']!r}")
print()
for h in report.details.manifest.history:
    print(f"  revision {h['revision']}: {h['action']}")

task now: regression
cv before: Cross-validate with StratifiedKFold (exact duplicates kept in one fold)
cv after:  Cross-validate with KFold (exact duplicates kept in one fold)
decisions removed: ['d.class_weight', 'd.task']
decisions added:   []
test set still registered: 'test set'

  revision 0: analyze
  revision 1: register_external_holdout
  revision 2: change_task


What changed:

- The task decision and the class-weighting decision are gone: there are no classes to weight.
- Cross-validation switched from `StratifiedKFold` to `KFold`; it still keeps duplicates in one fold.
- The test set registration survived, and the history records each step as a new **revision**. Every step returns a new report; the old one is unchanged.

### 4.2 Metric: which error matters?

In [13]:
show_questions(report)

[q.metric_priority] Which kind of error matters most in your application?
    'large_errors_costly'  Large errors are disproportionately costly
                           -> primary: RMSE
    'typical_error'        The typical (absolute) error matters most
                           -> primary: MAE
    'relative_error'       Errors relative to the true value matter
                           -> primary: MAPE


The metric question now offers regression metrics. prefit does not pick one, because the right metric depends on what an error costs in the application, and that is not in the data.

For a quality score the natural question is *"how many points off is the prediction, on average?"*. That is the mean absolute error (MAE). Nothing suggests that large errors are disproportionately costly (RMSE), and a relative error (MAPE) makes little sense for a score.

In [14]:
report = report.answer({"metric_priority": "typical_error"})
report.metric_plan

<MetricPlan primary=['neg_mean_absolute_error'] avoid=[]>

MAE is primary. Median absolute error, RMSE and R² are reported alongside it (see `report.metric_plan.report_alongside`).

## 5. The finding that matters here: exact duplicate rows
Three decisions are about exact duplicate rows. This is the finding that matters most on this dataset, so we read each one.

In [15]:
show_decision(report, "d.duplicate_split")
print()
show_decision(report, "d.cv")

Keep exact duplicates on one side of every split
  class: procedural   status: applied
  why:   195 extra exact copies would otherwise score their own twins.
  do:    Keep identical rows together in the holdout and in every CV fold.
  risk:  Near-duplicates (a rounding or one-column difference) are not caught; an id column makes copies distinct, so pass groups= if rows share an entity.
  evidence: Exact duplicate rows (extra copies) = 195
            (rows identical to an earlier row, all columns incl. target (15.26% of rows))

Cross-validate with KFold (exact duplicates kept in one fold)
  class: procedural   status: applied
  why:   Standard shuffled k-fold; exact duplicates (195 extra copies) are kept in one fold (rows hashed on features + target), so a copy never validates its twin.
  do:    Use KFold(n_splits=5, random_state=0, shuffle=True), keeping exact duplicates in one fold (prefit.DuplicateAwareKFold).
  risk:  Without groups or a time column, rows that belong together (same

**What a duplicate is here:** two rows with the same 11 measurements *and* the same score. prefit counts **195 extra copies** in the development rows, rows identical to an earlier row. (Counted another way, 375 development rows have a twin; both numbers are in the next cell.)

**Why it matters:** the test set exists to measure how the model does on wines it has **never seen**. If one copy lands in training and the other in the test set, the model has already seen that exact row; it does not predict it, it remembers it. The same happens between cross-validation folds.

**What prefit did:** both decisions are *procedural*, so they are applied without asking:

1. `pf.split` kept every group of copies on one side, which is why the overlap count in step 2 was **0**.
2. Cross-validation uses `pf.DuplicateAwareKFold`, which keeps every group of copies inside one fold. It hashes the rows itself, so we do not need to pass `groups=`.

No row was removed. Keeping copies together is correct whatever the copies are, so it needs no input from us.

What prefit does **not** decide is whether the copies should exist at all:

In [16]:
show_decision(report, "d.duplicate_rows")

Decide whether duplicates are the same record or legitimate repeats
  class: review   status: needs_review
  why:   195 extra exact copies (15.26%) weigh twice in training and scoring.
  do:    Decide whether they are the same record or legitimate repeats.
  risk:  Dropping legitimate repeats changes the data distribution; keeping artefacts over-weights them. prefit does not decide this for you.
  evidence: Exact duplicate rows (extra copies) = 195
            (rows identical to an earlier row, all columns incl. target (15.26% of rows))
  evidence: Rows in duplicate groups = 375 rows in 180 groups
            (every row that has an identical twin, and the number of distinct duplicated rows)
  evidence: Row distance between copies = 2.0
            (median rows from a copy to its first occurrence; random order gives 386 (permutation p = 0.005, 199 permutations))


This is a **review** item. The third piece of evidence is the interesting one: in the file, a copy sits a median of **2 rows** after its original, while random row order would put it about 386 rows away (permutation p = 0.005). That points to the same record entered more than once, not to two different wines that happen to share every value.

prefit says *points to*, not *proves*. Let us look at the data ourselves:

In [17]:
# Look at it ourselves: for each extra copy, how many rows above it is its first occurrence?
h = pd.Series(pd.util.hash_pandas_object(df, index=False).to_numpy())
first_pos = h.groupby(h).transform(lambda s: s.index[0])
is_copy = h.duplicated().to_numpy()
gaps = np.arange(len(df))[is_copy] - first_pos.to_numpy()[is_copy]

print(f"extra copies in the full data: {is_copy.sum()}")
print(f"distance to the first occurrence (rows): median {np.median(gaps):.0f}, max {gaps.max()}")
print(pd.Series(gaps).value_counts().sort_index().head(10).rename("copies at this distance").to_string())

i = int(np.flatnonzero(is_copy)[0])
print(f"\nExample: row {i} and the row it copies")
df.iloc[[int(first_pos.iloc[i]), i]]

extra copies in the full data: 240
distance to the first occurrence (rows): median 2, max 10
1     81
2     65
3     36
4     24
5     17
6      7
7      4
8      3
9      1
10     2

Example: row 4 and the row it copies


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
4,7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Every one of the 240 copies in the full file is at most 10 rows below its original, and most are 1–3 rows below. Rows 0 and 4, the pair we spotted in the first `head()`, are one example.

**Our call.** These are almost certainly re-entered records. We could drop the extra copies; we **keep** them, for two reasons: the dataset is used as published by many others, and since every split keeps copies together, they can no longer inflate a score. The remaining cost is the one in the *risk* line: the duplicated wines weigh twice in training.

A review item never enters the model recipe. `report.resolve` records our decision and our reason, so it stops being an open assumption and shows up, with the note, in the export, the review and the HTML report:

In [18]:
report = report.resolve(
    "d.duplicate_rows",
    "Copies sit next to their original in the file (re-entered records, most likely). Kept: every split keeps "
    "them together, so they cannot inflate a score; they weigh twice in training.",
)
show_decision(report, "d.duplicate_rows")

Decide whether duplicates are the same record or legitimate repeats
  class: review   status: reviewed
  why:   195 extra exact copies (15.26%) weigh twice in training and scoring.
  do:    Decide whether they are the same record or legitimate repeats.
  risk:  Dropping legitimate repeats changes the data distribution; keeping artefacts over-weights them. prefit does not decide this for you.
  your note: Copies sit next to their original in the file (re-entered records, most likely). Kept: every split keeps them together, so they cannot inflate a score; they weigh twice in training.
  evidence: Exact duplicate rows (extra copies) = 195
            (rows identical to an earlier row, all columns incl. target (15.26% of rows))
  evidence: Rows in duplicate groups = 375 rows in 180 groups
            (every row that has an identical twin, and the number of distinct duplicated rows)
  evidence: Row distance between copies = 2.0
            (median rows from a copy to its first occurrence; r

## 6. Standard practice
Two procedural decisions appear on every dataset. prefit flags them as standard practice and the reports list them apart, so the data-specific findings stand out:

In [19]:
for d in report.decisions:
    if d.standard:
        print(f"{d.id:15} {d.title}")
        print(f"{'':15} {d.why_short}")

d.holdout       Evaluate on the test set once, at the end
                Prefit did not reserve an internal holdout; the analysis ran on the frame you passed, and the registered test set (321 rows) was kept out of it.
d.fit_in_fold   Fit all preprocessing inside each fold
                Statistics fit on all rows leak test-fold information into the score.


## 7. A conditional decision: test it with our model

In [20]:
show_decision(report, "d.power_transform")

Power-transform 2 skewed + 2 near-threshold feature(s)
  class: conditional   status: proposed
  why:   2 feature(s) have |skew| >= 2 across their whole interval, and 2 more have intervals that straddle 2; whether that matters depends on the model family.
  do:    Test a Yeo-Johnson transform with your estimator (report.verify).
  risk:  Transforms make coefficients harder to interpret and can amplify noise in the tails.
  evidence: Skewness · residual sugar = 4.322177  95% CI [3.53, 4.8]
            (bootstrap 95% percentile interval (seeded))
  evidence: Skewness · chlorides = 5.740333  95% CI [4.87, 6.81]
            (bootstrap 95% percentile interval (seeded))
  evidence: Skewness · total sulfur dioxide = 1.604248  95% CI [1.14, 2.09]
            (bootstrap 95% percentile interval (seeded))
  evidence: Skewness · sulphates = 2.350844  95% CI [1.64, 2.86]
            (bootstrap 95% percentile interval (seeded))


How to read the evidence:

- Each skewness comes with a **95% bootstrap interval**, and the rule's threshold is |skew| ≥ 2.
- For `residual sugar` and `chlorides` the whole interval is above 2: the rule *fired*.
- For `total sulfur dioxide` and `sulphates` the interval contains 2, so the measurement is **near the threshold**. prefit says so instead of pretending to be sure. (`sulphates` has a skewness of 2.35, above the threshold, but its interval reaches down to 1.64.)

Whether a transform helps depends on the model, so prefit does not apply it; it tests it with the model we actually use.

**Our model:** a random forest, a strong and common default on small tabular data. `verify` compares *current setup* vs. *current setup + power transform* on identical folds of the same duplicate-aware splitter, repeated for statistical power.

In [21]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)

report = report.verify(dev_df, estimator=model)
report.decisions_table()[["class", "status", "decision", "experiment"]]

,class,status,decision,experiment
0,procedural,applied,Cross-validate with KFold (exact duplicates kept in one fold),
1,procedural,applied,Keep exact duplicates on one side of every split,
2,review,reviewed,Decide whether duplicates are the same record or legitimate repeats,
3,procedural,applied,Fit all preprocessing inside each fold,
4,conditional,no_practical_effect,Power-transform 2 skewed + 2 near-threshold feature(s),"user (RandomForestRegressor): MAE +0.0001662 (lower = better), 6/10 folds better, Holm p=0.9218, equivalence p=0.008..."
5,contextual,confirmed,Choose the metric from the cost of errors,
6,procedural,applied,"Evaluate on the test set once, at the end",


In [22]:
exp = report.experiments[0]
# Experiments store scikit-learn scores, where errors are negated ("neg_mean_absolute_error": higher = better).
# Flip the sign to read the change as an MAE.
sign = -1 if exp.metric.startswith("neg_") else 1
change, ci = sign * exp.mean_diff, sorted(sign * np.array(exp.ci))
direction = "lower = better" if sign < 0 else "higher = better"

print(f"cv:                  {exp.cv}")
print(f"MAE change:          {change:+.4f}   ({direction})")
print(f"95% CI:              [{ci[0]:+.4f}, {ci[1]:+.4f}]")
print(f"practical threshold: ±{exp.min_effect:.4f}   (1% of the baseline score)")
print(f"difference test:     Holm p = {exp.p_adjusted:.3f}")
print(f"equivalence test:    Holm p = {exp.p_equivalence:.2g}")
print(f"folds better:        {exp.folds_improved} / {len(exp.baseline_scores)}")
print(f"outcome:             {exp.outcome}")
print()
show_decision(report, "d.power_transform")

cv:                  DuplicateAwareKFold(n_splits=5, n_repeats=2, stratify=False, random_state=0) — the export splitter, repeated; exact duplicates kept in one fold
MAE change:          +0.0002   (lower = better)
95% CI:              [-0.0036, +0.0039]
practical threshold: ±0.0050   (1% of the baseline score)
difference test:     Holm p = 0.922
equivalence test:    Holm p = 0.0082
folds better:        6 / 10
outcome:             no_practical_effect

Power-transform 2 skewed + 2 near-threshold feature(s)
  class: conditional   status: no_practical_effect
  why:   2 feature(s) have |skew| >= 2 across their whole interval, and 2 more have intervals that straddle 2; whether that matters depends on the model family.
  do:    Test a Yeo-Johnson transform with your estimator (report.verify).
  risk:  Transforms make coefficients harder to interpret and can amplify noise in the tails.
  note:  Equivalent within the practical threshold (TOST, Holm-adjusted): the change makes no practical differ

Three tests, three answers:

- **Is there a difference?** No. The Holm-adjusted p-value is 0.92, and the change in MAE (+0.0002) is tiny next to its interval.
- **Is the difference large enough to matter?** prefit's practical threshold is 1% of the baseline score, ±0.005 MAE here.
- **Is there evidence that there is *no* practical difference?** Yes. The equivalence test (two one-sided tests against ±0.005) gives p = 0.008: the whole interval sits inside the practical band.

So the status is **no practical effect**, and the step stays out of the model recipe. This is different from *inconclusive*, which prefit uses when the folds are too noisy to tell either way. Absence of evidence is not reported as evidence of absence.

That a monotone transform does nothing for a tree ensemble is the expected result, but prefit did not assume it: it measured it, with our model.

## 8. Export the model recipe and train

In [23]:
built = report.export()
print(built.describe())

EXPORTED MODEL RECIPE
model: Pipeline(preprocess, estimator)
features: 11 columns
cv: DuplicateAwareKFold(n_splits=5, n_repeats=1, stratify=False, random_state=0)
scoring: ['neg_mean_absolute_error', 'neg_median_absolute_error', 'neg_root_mean_squared_error', 'r2']
estimator-dependent default: StandardScaler on numeric features (scale=False to turn off)
cv keeps exact duplicate rows (features + target) in one fold; no groups= needed
included:
  (none)
not included:
  - d.power_transform [no_practical_effect]: experiment shows no practical effect (equivalent within the practical threshold)
review items you closed:
  ✓ Decide whether duplicates are the same record or legitimate repeats: Copies sit next to their original in the file (re-entered records, most likely). Kept: every split keeps them together, so they cannot inflate a score; they weigh twice in training.


`export()` builds a scikit-learn recipe that contains **only** applied, confirmed or verified steps, and lists what was left out and why.

- The model is a plain `Pipeline(preprocess, estimator)`. The scaler is an estimator-dependent default, not a decision (`scale=False` turns it off; it does not matter for a forest).
- `cv` is the same duplicate-aware splitter the experiments used, ready for `cross_validate(model, X, y, cv=built.cv)`.
- The power transform is listed as *not included*, with its status; our review note is listed under *review items you closed*.

`report.cv_estimate()` gives the development cross-validation estimate from `verify` in natural units. We fit the recipe on all development rows:

In [24]:
cv = report.cv_estimate()
print(f"development CV estimate: {cv['label']} {cv['mean']:.3f} ± {cv['std']:.3f}  ({cv['n_folds']} folds)")

X_cols = built.feature_columns
fitted = built.model.fit(dev_df[X_cols], dev_df[TARGET])

development CV estimate: MAE 0.501 ± 0.032  (10 folds)


## 9. Final evaluation on the untouched test set
We now use `test_df` for the first and only time.

Besides the regression metrics, two numbers that are easier to read for a score: **exact score**, the share of wines whose rounded prediction is exactly right, and **within ±1**, the share that is at most one point off. As a reference, a model that always predicts the median score of the development data.

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def score_table(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred, dtype=float)
    rounded = np.clip(np.rint(y_pred), y_true.min(), y_true.max())
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "exact score": np.mean(rounded == y_true),
        "within ±1": np.mean(np.abs(rounded - y_true) <= 1),
    }


y_test = test_df[TARGET]
pred = fitted.predict(test_df[X_cols])
median_pred = np.full(len(y_test), dev_df[TARGET].median())

scores = pd.DataFrame({
    "random forest (prefit recipe)": score_table(y_test, pred),
    "always predict the median": score_table(y_test, median_pred),
}).round(3)
scores

,random forest (prefit recipe),always predict the median
MAE,0.467,0.660
RMSE,0.629,0.890
R2,0.401,-0.198
exact score,0.607,0.399
within ±1,0.969,0.947


The forest gets 61% of the scores exactly right and 97% within one point, against 40% and 95% for the median. MAE drops from 0.66 to 0.47.

Is 0.467 *the* score of this model? A single test set of 321 wines is a noisy measurement. `pf.review` checks the roadmap's assumptions against the real errors, puts a **bootstrap interval** on every headline metric, and checks the development CV estimate against that interval. Passing the test features as `X=` lets the bootstrap resample exact copies together: the test set contains 45 extra copies of its own, and treating them as independent would make the interval look tighter than it is.

In [26]:
review = pf.review(y_test, pred, report, X=test_df[X_cols])
print(review)

REVIEW — did the roadmap hold?
Task: regression · n=321

Measured:
- MAE: 0.4671  [95% CI 0.4172, 0.5188]
- RMSE: 0.6289  [95% CI 0.5646, 0.6957]
- median_abs_error: 0.34
- R2: 0.4014  [95% CI 0.2971, 0.4872]
- RMSE_MAE_ratio: 1.346
- worst5pct_error_share: 0.1859
- baseline MAE (predict the median): 0.6604

Checks:
- [OBSERVED] Error-distribution shape (no skew warning was raised).
    RMSE/MAE=1.35; worst 5% carry 19%. Errors look well-behaved.
    reference: Gaussian errors give RMSE/MAE = 1.25 and worst-5% share = 0.15; 'tails dominate' = ratio >= 1.2x or share >= 2x those.
- [HELD] The model beats a constant (median) prediction.
    MAE 0.4671 vs 0.6604 (+29.3%).
    reference: MAE of always predicting the median of y_true
- [HELD] The development CV estimate describes the score on the untouched test data.
    Held: CV MAE 0.5015 lies inside the test interval [0.4172, 0.5188]; the gap to the test score (0.4671) is within test-set noise.
    reference: 95% bootstrap interval of the

How to read this:

- The test MAE is 0.467, with a 95% interval of about **[0.42, 0.52]**. That is the honest precision of one test set of this size.
- The development CV estimate, 0.501, lies **inside** that interval. The check *held*: what we knew before touching the test set describes the test result.
- The CV estimate is 0.03 higher than the test score. That gap is within test-set noise, and it would be a mistake to read a story into it (step 10 shows that over ten different splits the gap averages out to about zero).
- Our review note on the duplicates is listed; the power transform is no longer an open assumption, because the experiment settled it.

## 10. Without prefit: a plain scikit-learn workflow
What would the same model report in a typical workflow? A stratified `train_test_split` on the raw data, `KFold` cross-validation on the training part, the same random forest and the same seed. The only difference is that nothing keeps exact duplicates together.

In [27]:
from sklearn.model_selection import KFold, cross_val_score, train_test_split

feature_cols = [c for c in df.columns if c != TARGET]

plain_train, plain_test = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=SEED)
plain_model = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)

plain_cv_mae = -cross_val_score(
    plain_model, plain_train[feature_cols], plain_train[TARGET],
    cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error",
).mean()
plain_pred = plain_model.fit(plain_train[feature_cols], plain_train[TARGET]).predict(plain_test[feature_cols])
plain_test_mae = mean_absolute_error(plain_test[TARGET], plain_pred)

print(f"plain CV estimate: MAE {plain_cv_mae:.3f}")
print(f"plain test set:    MAE {plain_test_mae:.3f}")

plain CV estimate: MAE 0.429
plain test set:    MAE 0.417


The plain workflow reports a clearly better MAE: 0.417 on its test set against 0.467 for prefit, and 0.429 against 0.501 in cross-validation. Is the model better? It is the same model. Split its test set into rows that have an exact copy in its training data and rows that do not:

In [28]:
row_hash = pd.util.hash_pandas_object(df, index=False)
seen = row_hash.loc[plain_test.index].isin(set(row_hash.loc[plain_train.index])).to_numpy()
y_plain = plain_test[TARGET].to_numpy()

print(f"plain test rows with a copy in training: {seen.sum()} / {len(seen)} ({seen.mean():.1%})")
print(f"  MAE on those rows:     {mean_absolute_error(y_plain[seen], plain_pred[seen]):.3f}")
print(f"  MAE on the other rows: {mean_absolute_error(y_plain[~seen], plain_pred[~seen]):.3f}")

plain test rows with a copy in training: 76 / 320 (23.8%)
  MAE on those rows:     0.157
  MAE on the other rows: 0.498


About a quarter of the plain test set (76 of 320 rows) had been seen in training. On those rows the error is 0.157: the forest recalls them. On the rows it had never seen, the error is 0.498, in line with prefit's CV estimate.

One split can be lucky or unlucky, as step 9 showed. To make sure the comparison does not rest on seed 42, we repeat both workflows over ten different splits (with a smaller forest, to keep it fast):

In [29]:
def one_seed(seed, n_trees=150):
    rf = lambda: RandomForestRegressor(n_estimators=n_trees, random_state=0, n_jobs=-1)  # noqa: E731
    out = {"seed": seed}
    # plain: row-level split and row-level KFold
    tr, te = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=seed)
    out["plain CV"] = -cross_val_score(rf(), tr[feature_cols], tr[TARGET], cv=KFold(5, shuffle=True, random_state=0),
                                       scoring="neg_mean_absolute_error").mean()
    out["plain test"] = mean_absolute_error(te[TARGET], rf().fit(tr[feature_cols], tr[TARGET]).predict(te[feature_cols]))
    # prefit: duplicate-aware split and duplicate-aware CV
    dv, ts = pf.split(df, TARGET, test_size=0.2, random_state=seed)
    out["prefit CV"] = -cross_val_score(rf(), dv[feature_cols], dv[TARGET], cv=pf.DuplicateAwareKFold(5, 1, False, 0),
                                        scoring="neg_mean_absolute_error").mean()
    out["prefit test"] = mean_absolute_error(ts[TARGET], rf().fit(dv[feature_cols], dv[TARGET]).predict(ts[feature_cols]))
    return out


seeds = pd.DataFrame([one_seed(s) for s in range(10)]).set_index("seed")

pd.DataFrame({
    wf: {
        "CV estimate (mean)": seeds[f"{wf} CV"].mean(),
        "test MAE (mean)": seeds[f"{wf} test"].mean(),
        "test MAE (std over seeds)": seeds[f"{wf} test"].std(),
        "test minus CV (mean)": (seeds[f"{wf} test"] - seeds[f"{wf} CV"]).mean(),
    }
    for wf in ("plain", "prefit")
}).round(3)

,plain,prefit
CV estimate (mean),0.438,0.495
test MAE (mean),0.409,0.493
test MAE (std over seeds),0.025,0.014
test minus CV (mean),-0.029,-0.002


Over ten splits:

- The plain workflow's test MAE averages about **0.41**; prefit's about **0.49**. The plain number is about **17% too optimistic**, and that holds across splits, not just for one seed.
- prefit's CV estimate and its test score agree on average (the difference is about zero). The plain workflow's test score is even better than its own CV estimate, because the final model is trained on more rows and therefore has seen more of the test set's copies.
- The model is the same in both workflows. prefit did **not** make it more accurate: the one preprocessing step it proposed had no practical effect. What it changed is that the reported number can be trusted.

## 11. Save the HTML report
The full report has four sections: summary, actions, a roadmap card per decision (with evidence, experiments and figures one click away, and standard practice folded away), and details. We embed the final review in it. `print(report)` gives the same roadmap as text.

`plotlyjs="inline"` puts plotly.js into the file (about 5 MB) so the charts work offline. On Kaggle, the file appears under **Output** and can be downloaded.

In [30]:
report_path = "/kaggle/working/prefit_wine_report.html" if glob.glob("/kaggle/working") else "prefit_wine_report.html"
report.to_html(report_path, review=review, plotlyjs="inline")
print(f"wrote {report_path}")

wrote prefit_wine_report.html


## 12. Does an LLM help?
Everything so far was deterministic: fixed rules over measured statistics. The rules know nothing about the *problem*: what a wine is, how the score was produced, what the columns mean. That knowledge entered only through our answers.

prefit has an optional LLM layer for this gap, and it is deliberately boxed in:

- It never sees raw rows or category labels, only the computed report (column names and kinds, statistics, findings, decisions, open questions) and a free-text context we write.
- It can summarise, prioritise existing decisions, **suggest** answers to open questions, ask us questions, and raise semantic flags about **column names**.
- It cannot add findings or decisions, compute statistics, or apply anything. Claims that cite ids not in the report are dropped (and listed). Suggestions stay "awaiting confirmation" until we call `answer()`.

**The experiment.** We give the LLM the report *before* any of our answers, plus a one-paragraph description of the problem that does not hint at them. Then we give it the report *after* our answers. We check two things: does it build advice on the guessed task, and does it add anything we did not know? The model is Claude Sonnet 5.5.

> **On Kaggle:** Add-ons → Secrets → add `ANTHROPIC_API_KEY` and attach it to this notebook. Without a key the cells below still run and say that no interpretation came back.

In [ ]:
%pip install -q --disable-pip-version-check "prefit[llm]>=0.3.1"

In [32]:
import os

try:  # on Kaggle: Add-ons -> Secrets -> ANTHROPIC_API_KEY
    from kaggle_secrets import UserSecretsClient

    os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
except Exception:
    pass  # not on Kaggle, or no secret attached: set ANTHROPIC_API_KEY in the environment

HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("API key found:", HAVE_KEY)  # the key itself is never printed

API key found: True


### What leaves the machine

The context is what a user would write: what the data is and what the model is for. It deliberately says nothing about the task type, the metric or the duplicates.

`llm_payload()` returns exactly what `interpret` would send, without sending anything. Read it before any API call.

In [33]:
CONTEXT = (
    "Each row is a red Vinho Verde wine from northern Portugal. The 11 columns are physicochemical lab "
    "measurements taken at certification. 'quality' is the median of at least three expert tasters' scores on "
    "a 0-10 scale. Goal: predict the tasters' score from the lab measurements, so that wines can be "
    "pre-screened before a tasting session."
)

# a fresh analysis, before any of our answers: task and metric still open
open_report = pf.analyze(dev_df, target=TARGET, context=CONTEXT)
open_report = open_report.register_external_holdout(test_df, dev_df=dev_df, name="test set")

LLM = pf.LLMConfig(provider="anthropic", model="claude-sonnet-5-5", max_tokens=1500)
payload = open_report.llm_payload(LLM)

print(f"fields sent:     {list(payload)}")
print(f"payload size:    {len(str(payload)):,} characters")
print(f"task confirmed:  {payload['task_confirmed']}")
print("marked task-dependent:")
for item in payload["findings"] + payload["decisions"]:
    if item.get("task_dependent"):
        print(f"  {item['id']}")

fields sent:     ['task', 'task_inferred', 'task_confirmed', 'n_rows', 'n_features', 'target', 'columns', 'evidence', 'findings', 'decisions', 'questions', 'task_reason', 'user_context']
payload size:    7,718 characters
task confirmed:  False
marked task-dependent:
  f.class_imbalance
  d.class_weight
  d.metric


The payload says that the task is **not confirmed**, and marks the three items that exist only under the guessed task (the class-imbalance finding, class weights, and the metric options). The system prompt tells the model not to build advice on them. Let us see whether it follows that.

In [34]:
import textwrap


def wrap(text, indent="    "):
    return textwrap.fill(str(text), width=110, initial_indent=indent, subsequent_indent=indent)


def show_llm(rep):
    out = rep.details.llm
    if out is None:
        print("No interpretation came back; see the warning above.")
        return
    print(f"model: {out.model}   dropped items: {out.dropped_claims}")
    for x in out.dropped:
        print(wrap(f"dropped: {x}"))
    print("\nSUMMARY")
    print(wrap(out.summary))
    print("\nPRIORITIES")
    for c in out.priorities:
        print(wrap(f"- {c.text}"))
        print(wrap(f"cites: {list(c.decision_ids) + list(c.evidence_ids)}", indent="        "))
    print("\nSUGGESTED ANSWERS (not applied)")
    sug = [(q.id, q.suggestion, q.suggestion_reason) for q in rep.questions if q.suggestion]
    for qid, s, why in sug:
        print(wrap(f"- {qid}: {s!r} — {why}"))
    if not sug:
        print("    (none)")
    print("\nQUESTIONS FOR US")
    for t in out.questions_for_user:
        print(wrap(f"- {t}"))
    print("\nSEMANTIC FLAGS ON COLUMN NAMES")
    for f in out.semantic_flags:
        print(wrap(f"- {f.column}: {f.concern} ({f.reason})"))
    if not out.semantic_flags:
        print("    (none)")


interpreted_open = open_report.interpret(LLM) if HAVE_KEY else open_report
show_llm(interpreted_open)

model: claude-sonnet-5-5   dropped items: 0

SUMMARY
    Red Vinho Verde dataset: 1278 development rows, 11 numeric lab features, and a 321-row test set registered
    as the external holdout. The task is only inferred as classification, so it must be confirmed before any
    class-based advice is acted on. Duplicates are the main data-quality concern: 195 extra copies, sitting
    close together in the file. Duplicate-aware CV is already applied. Several features are skewed and a Yeo-
    Johnson transform is proposed.

PRIORITIES
    - Confirm the task first. The target is an integer 0-10 expert score with 6 distinct values, which could
    be class labels or an ordered score. Imbalance, class weighting and metric choices exist only under the
    classification guess, so I am not building advice on them yet.
        cites: ['d.task', 'ev.task.inference']
    - Resolve the duplicate-row review. 195 extra exact copies (375 rows in 180 groups) sit about 2 rows
    apart, which points to

**What it did with an unconfirmed task** *(this run; LLM output varies between runs, so read the cell above, not only this text)*:

- Its first priority is *"Confirm the task first"*: imbalance, class weighting and the metric *"exist only under the classification guess, so I am not building advice on them yet"*. None of its priorities is built on a task-dependent item (table below). In an earlier run with prefit 0.2.0, whose payload did not mark these items, the same model made the critical class imbalance its top priority.
- It did not suggest a task. That is what the prompt asks for: our context describes the score but does not settle how to model it.
- It read the duplicate evidence correctly, including that copies sit about two rows apart, and asked us to record a decision.
- It described the power transform as *proposed*, which is accurate: nothing had been tested yet.
- Its metric question is explicitly conditional (*"Once the task is confirmed as classification: ..."*), so the guessed task shaped a question but not an answer.

No semantic flags, and no domain knowledge about the columns (for example that `free sulfur dioxide` is part of `total sulfur dioxide`).

In [35]:
def framing(rep):
    # Which decisions and evidence do the LLM's priorities cite, and how many of them only exist under the guessed task?
    out = rep.details.llm
    if out is None:
        return {}
    dep = {d.id for d in rep.decisions if d.task_dependent} | {
        e for f in rep.details.findings if f.task_dependent for e in f.evidence_ids}
    cited = [x for c in out.priorities for x in (*c.decision_ids, *c.evidence_ids)]
    sug = {q.id: q.suggestion for q in rep.questions if q.suggestion}
    confirmed = rep.llm_payload()["task_confirmed"]
    pt = rep.decision("d.power_transform").status.value
    return {
        "task confirmed in the payload": confirmed,
        "priorities": len(out.priorities),
        "priorities built on the guessed task": "n/a (task confirmed)" if confirmed else sum(
            any(x in dep for x in (*c.decision_ids, *c.evidence_ids)) for c in out.priorities),
        "power transform status it saw": pt,
        "priorities citing the power transform": sum("d.power_transform" in c.decision_ids for c in out.priorities),
        "suggested task": sug.get("q.task", "none"),
        "suggested metric": sug.get("q.metric_priority", "none"),
        "semantic flags": len(out.semantic_flags),
        "dropped items": out.dropped_claims,
        "cites duplicates": any("duplicate" in x for x in cited),
    }


pd.Series(framing(interpreted_open), name="before our answers").to_frame()

,before our answers
task confirmed in the payload,False
priorities,3
priorities built on the guessed task,0
power transform status it saw,proposed
priorities citing the power transform,1
suggested task,none
suggested metric,none
semantic flags,0
dropped items,0
cites duplicates,True


Now the same analysis **after** our answers: regression, MAE, the review note, and the experiment with our forest. Does the LLM respect what we decided and what the experiment found?

A review item we closed with `resolve` is sent as `closed_by_user`, with our note. The note is free text, so it follows the same rule as the context: it is only sent with `include_context=True` and shared column names.

In [36]:
# the same steps as 4-7 (task, metric, review note, experiment), now with the context attached
final_with_context = pf.analyze(dev_df, target=TARGET, task="regression", context=CONTEXT,
                                answers={"metric_priority": "typical_error"})
final_with_context = final_with_context.register_external_holdout(test_df, dev_df=dev_df, name="test set")
final_with_context = final_with_context.resolve("d.duplicate_rows", report.decision("d.duplicate_rows").review_note)
final_with_context = final_with_context.verify(dev_df, estimator=model)

for item in final_with_context.llm_payload(LLM)["decisions"]:
    if item.get("closed_by_user"):
        print(f"sent as closed: {item['id']}  user_note: {item.get('user_note')!r}\n")

interpreted_final = final_with_context.interpret(LLM) if HAVE_KEY else final_with_context
show_llm(interpreted_final)

sent as closed: d.duplicate_rows  user_note: 'Copies sit next to their original in the file (re-entered records, most likely). Kept: every split keeps them together, so they cannot inflate a score; they weigh twice in training.'



model: claude-sonnet-5-5   dropped items: 0

SUMMARY
    Regression on 1278 red Vinho Verde wines (11 numeric lab features, target 'quality'), with the task
    confirmed. The main data issue is 195 extra exact duplicate rows (15.26%). You have already decided to
    keep them, and every split keeps them together. Four features are skewed or near the skew threshold, but
    the Yeo-Johnson transform was judged to have no practical effect. MAE is the confirmed primary metric. A
    321-row test set is registered and will be scored once at the end.

PRIORITIES
    - Duplicates are handled by your decision: kept, and held together in every CV fold and in the holdout, so
    they cannot inflate scores. Just be aware they carry double weight in training.
        cites: ['d.duplicate_rows', 'd.duplicate_split', 'd.cv', 'ev.dataset.duplicates',
        'ev.dataset.duplicate_groups', 'ev.dataset.duplicate_adjacency']
    - Fit all preprocessing inside each CV fold using a single Pipeline, and 

In [37]:
pd.DataFrame({"before our answers": framing(interpreted_open), "after our answers": framing(interpreted_final)})

,before our answers,after our answers
task confirmed in the payload,False,True
priorities,3,4
priorities built on the guessed task,0,n/a (task confirmed)
power transform status it saw,proposed,no_practical_effect
priorities citing the power transform,1,1
suggested task,none,none
suggested metric,none,none
semantic flags,0,0
dropped items,0,0
cites duplicates,True,True


**After our answers:**

- It carried our closed review item correctly: *"Duplicates are handled by your decision: kept, and held together in every CV fold and in the holdout ... they carry double weight in training."* It did not reopen it, and it asked no questions at all (the empty *questions for us* section).
- It respected the experiment: the power transform *"was assessed as having no practical effect, so it is low priority"*.
- It kept the confirmed metric and the evaluation protocol.

> This guide found a gap here in prefit 0.3.0: a review item closed with `resolve` reached the LLM without its note, and the model presented it as still open. prefit 0.3.1 sends closed items as `closed_by_user` with the note, which is what this run used.

**Verdict for this dataset.** The LLM step was safe and cheap: it invented nothing (0 dropped items), it kept to the guard rails around the unconfirmed task, and it deferred to our decisions and to the experiment. But it added nothing the report did not already contain: no domain knowledge, no new question that prefit's own task question does not already ask. On this dataset its value is a readable summary, not new insight, and prefit still applies nothing it says without `answer()`.

## Summary

In [38]:
print("What this run measured")
print(f"  exact duplicates:            {int(df.duplicated(keep=False).sum())} of {len(df)} rows have an identical twin")
print(f"  test rows seen in training:  prefit 0, plain workflow {seen.sum()} of {len(seen)}")
print(f"  development CV (prefit):     MAE {cv['mean']:.3f} ± {cv['std']:.3f}")
lo, hi = review.intervals["MAE"]
print(f"  test set (prefit):           MAE {scores.loc['MAE', 'random forest (prefit recipe)']:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print(f"  plain workflow:              CV {plain_cv_mae:.3f}, test {plain_test_mae:.3f}")
print(f"  over 10 seeds:               plain test {seeds['plain test'].mean():.3f}, prefit test {seeds['prefit test'].mean():.3f}"
      f" (plain is {1 - seeds['plain test'].mean() / seeds['prefit test'].mean():.0%} lower)")
print(f"  power transform:             {report.decision('d.power_transform').status.value}")

What this run measured
  exact duplicates:            460 of 1599 rows have an identical twin
  test rows seen in training:  prefit 0, plain workflow 76 of 320
  development CV (prefit):     MAE 0.501 ± 0.032
  test set (prefit):           MAE 0.467  95% CI [0.417, 0.519]
  plain workflow:              CV 0.429, test 0.417
  over 10 seeds:               plain test 0.409, prefit test 0.493 (plain is 17% lower)
  power transform:             no_practical_effect


**What prefit found on a "clean" dataset**

- **Duplicates were the finding that mattered.** 460 of the 1,599 rows have an identical twin, and the copies sit next to their originals in the file. A plain split put copies on both sides: about a quarter of its test set had been seen in training, and its MAE was about 17% too optimistic over ten splits. prefit kept copies together in the split and in every CV fold without removing a row, gave the evidence that they are re-entered records, and left the decision to us. We recorded it with `resolve`.
- **The question with the largest effect could not come from the data.** Classification or regression decided which findings existed at all, including a "critical" class imbalance that only exists if the score is a set of unordered classes. prefit flagged those findings as task-dependent, asked, and `change_task` applied the answer without losing anything else.
- **The proposed preprocessing was tested, not assumed.** The power transform had *no practical effect* on our forest (equivalence shown, not just "no significant difference"), so it stayed out of the recipe.
- **The estimate before the test set was trustworthy.** The CV estimate (0.501) lay inside the test score's 95% interval [0.42, 0.52], and over ten splits prefit's CV and test scores agreed on average.

**Where prefit did not help**

- The final model is the same random forest we would have trained anyway; no step improved its accuracy. The gain is entirely in the evaluation.
- The LLM layer added no knowledge beyond the report.

**Limitations to keep in mind**

- Only *exact* duplicates are matched. Near-duplicates (the same wine measured with rounding differences) are not detected; the adjacency test is a hint, not proof.
- Leakage screening sees single-feature association only.
- Experiments test one change at a time; interactions between decisions are not measured.

**Links:** [GitHub](https://github.com/hkln4/prefit) · [PyPI](https://pypi.org/project/prefit/) · [Project page](https://hkln4.github.io/prefit/) · [Changelog](https://github.com/hkln4/prefit/blob/main/CHANGELOG.md)